# Unidad 2: Aprendizaje Automático 
## Regresión Lineal con Scikit-learn II
### Inteligencia Artificial - Lic. en Sistemas - FCAD/UNER

![Store](https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/notebooks/ml/images/pexels-ja-kubislav-374578277-14534753.jpg)

[Foto de Ja Kubislav](https://www.pexels.com/es-es/foto/moda-colorido-de-colores-ventanas-14534753/)

---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CristianPacifico/ia-ls-fcad-uner/blob/main/notebooks/ml/03_LinealRegression_Ecommerce.ipynb)


## Dataset
Una compañía de E-commerce vende ropa online, pero tienen una tienda física donde pueden venir a recibir asesoría, consejos de estilos. Intentan identificar si debe enfocarse en mejorar su experiencia de app movil o en su página web, para ello se detalla las características que son la información de los clientes:

* Avatar: Color de Avatar.
* Avg. Session Length: Average session of in-store style advice sessions.
* Time on App: Tiempo promedio que pasa en la app en minutos.
* Time on Website: Tiempo promedio que pasa en el sitio web en minutos.
* Length of Membership: Cuantos años el cliente ha sido cliente.
* Yearly Amount Spent

# Librerías

Importamos las librerías que usaremos, ***pandas*** para el manejo de la data, ***numpy*** para operaciones aritméticas,
***matplotlib.pyplot*** y ***seaborn*** para los gráficos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn import metrics

**traint_test_split** nos servirá para dividir nuestro dataset

***LinearRegression*** nos servirá para operar con la regresión lineal

***metrics*** nos servirá para obtener las métricas de nuestra predicción

# Manejo de la Data

Cargamos nuestra data en la variable ***df***

In [ ]:
df=pd.read_csv('https://raw.githubusercontent.com/CristianPacifico/ia-ls-fcad-uner/main/data/ml/Ecommerce%20Customers.csv')

Obtenemos una vista previa de nuestra data

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
df.info()

Por aplicar esta data en un caso de regresión lineal, se eliminan las columnas de tipo 'object' (como 'Email', 'Address', 'Avatar') porque estos datos son categóricos o de texto y no pueden ser utilizados directamente por el modelo, que solo acepta variables numéricas. 

Incluir columnas de tipo 'object' sin transformarlas puede causar errores o resultados incorrectos, ya que la regresión lineal requiere variables cuantitativas para calcular  relaciones matemáticas. Por eso, solo se usan columnas numéricas como _predictores_.

In [ ]:
df.drop(columns=['Email', 'Address', 'Avatar'], inplace=True)

In [ ]:
df.info()

In [ ]:
df.describe()


# Análisis visual de la data

In [ ]:
sns.pairplot(df)

In [ ]:
sns.heatmap(df.corr(), annot=True)

Se observa que hay mayor correlación entre *Lenght of Membership* y *Yearly Amount Spent*

Esto quiere decir que mientras más antiguo es un cliente, este tiende a gastar más al año.

In [ ]:
sns.lmplot(x='Length of Membership',y='Yearly Amount Spent',data=df)
plt.show()

# Regresión lineal

Para empezar crearemos un dataframe ***X*** que contendrá las columnas *Avg. Session Length*, *Time on App*, *Time on Website* y *Length of Membership*.

Luego crearemos otro dataframe ***y*** que contenga la columna *Yearly Amount Spent*.


In [ ]:
X = df[['Avg. Session Length', 'Time on App', 'Time on Website', 'Length of Membership']]
X

In [ ]:
y= df['Yearly Amount Spent']
y

Ahora dividiremos estos 2 dataframes en sus respectivos train/test con la función **train_test_split()**, la proporción será de 70% / 30% y la semilla que usaremos es 10.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=10)

Lo que sigue es crear un objeto de tipo **LinearRegression** para hacer uso de los métodos:

-**fit()** para adecuar nuestra data al modelo de regresión lineal

-**coef_** para ver los coeficientes de nuestro modelo


In [ ]:
lm = LinearRegression()

In [ ]:
lm.fit(X_train,y_train)

In [ ]:
lm.coef_

Una vez hecho esto tenemos listos el 'peso' de cada atributo con relación a lo que gasta un cliente al año.

In [ ]:
resultados = pd.DataFrame(lm.coef_,X.columns,columns=['Coeff'])

In [ ]:
resultados

¿Qué significan estas métricas?

-Una unidad de *Avg. Session Lenght* se asocia a un aumento de 25.8 en lo que gasta el cliente al año.

-Una unidad de *Time on App* se asocia a un aumento de 38.8 en lo que gasta el cliente al año.

-Una unidad de *Time on Website* se asocia a un aumento de 0.96 en lo que gasta el cliente al año.

-Una unidad de *Lenght of Membership* se asocia a un aumento de 61.8 en lo que gasta el cliente al año.


# Predicción

Lo que sigue es hacer una predicción de nuestra data de prueba para ver si se asemeja al escenario real. 

In [ ]:
predic = lm.predict(X_test)

En la siguiente celda se realiza un gráfico de dispersión entre los valores reales de gasto anual (`y_test`) y los valores predichos por el modelo (`predic`). 

Esto permite visualizar qué tan bien el modelo de regresión lineal predice los datos reales: si los puntos se alinean cerca de una línea recta, indica que las predicciones son precisas.

In [ ]:
plt.scatter(y_test,predic)
plt.xlabel('Y Test (valores reales)')
plt.ylabel('valores predichos')

En la siguiente celda se graficará la diferencia entre los valores reales y los valores predichos (residuos) utilizando un histograma. 

Esto permite visualizar la distribución de los errores del modelo y evaluar si siguen una distribución normal, lo cual es un buen indicador de que el modelo realiza buenas predicciones.

In [ ]:
sns.histplot(y_test-predic, kde=True, stat="density")

El modelo tiene forma de campana, lo que quiere decir que se hicieron buenas predicciones. 

# Métricas

Para finalizar veremos qué tan preciso fue nuestro modelo observando las siguientes métricas:

- **R² (Coeficiente de determinación):** Indica qué proporción de la variabilidad de la variable objetivo (Yearly Amount Spent / Monto anual gastado) es explicada por el modelo. Un valor cercano a 1 significa que el modelo explica bien los datos.

- **MAE (Mean Absolute Error):** Es el promedio de las diferencias absolutas entre los valores reales y los predichos. Mide el error promedio en las mismas unidades que la variable objetivo (en dólares/pesos).

- **MSE (Mean Squared Error):** Es el promedio de los errores al cuadrado entre los valores reales y los predichos. Penaliza más fuertemente los errores grandes.

- **RMSE (Root Mean Squared Error):** Es la raíz cuadrada del MSE. Representa el error promedio en las mismas unidades que la variable objetivo y es más interpretable que el MSE.

In [ ]:
print ('R²:',metrics.r2_score(y_test,predic))
print ('MAE:',metrics.mean_absolute_error(y_test,predic))
print ('MSE:', metrics.mean_squared_error(y_test,predic))
print ('RMSE', np.sqrt(metrics.mean_squared_error(y_test,predic)))

# Conclusión

La empresa de E-commerce tiene dos opciones:

-Mejorar la experiencia en la Aplicación para que los clientes estén más tiempo en ella.

-Mejorar la experiencia en el sitio web para que sea tan eficiente como la aplicación.


-Opción adicional: Mejorar la relación con los clientes para que sean miembros por un largo periodo de tiempo.

---

© 2026 Cátedra Inteligencia Artificial — Lic. en Sistemas

Este notebook se distribuye bajo licencia [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).